In [1]:
!pip install -q skl2onnx onnxruntime kagglehub supabase

In [13]:
import glob
import os
import re
import kagglehub
import pandas as pd

# 1. Load Philippine Spam SMS dataset from Kaggle
ph_path = kagglehub.dataset_download("bwandowando/philippine-spam-sms-messages")
csv_files = glob.glob(os.path.join(ph_path, "*.csv"))
df_ph = pd.read_csv(csv_files[0])

# Auto-detect the message text column (picks the column with longest average text)
text_col = max(
    df_ph.select_dtypes(include="object").columns,
    key=lambda c: df_ph[c].astype(str).str.len().mean(),
)
df_ph_spam = pd.DataFrame({"text": df_ph[text_col].dropna(), "label": 1})

# 2. Load legitimate (Ham) messages from UCI SMS Spam Collection
uci_url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df_uci = pd.read_csv(uci_url, sep="\t", names=["label_str", "text"])
df_uci_ham = pd.DataFrame(
    {"text": df_uci[df_uci["label_str"] == "ham"]["text"], "label": 0}
)

# 3. Add local Philippine Ham examples so real alerts aren't flagged as spam
ph_ham_samples = [
    "Your GCash OTP is 482910. Never share this code with anyone.",
    "You have received PHP 500.00 in your GCash account from JUAN DELA CRUZ.",
    "BDO Online: Your One-Time PIN is 918273. Valid for 5 minutes.",
    "Shopee: Your parcel with tracking PH29384712 is out for delivery today.",
    "Lazada: Rider will arrive soon. Please prepare exact amount PHP 349.00.",
    "GrabFood: Your driver is on the way to your location.",
    "Globe: Your Go+99 promo is now active. Enjoy 8GB all-access data for 7 days.",
    "Smart: You have used 80% of your GigaPower data allocation.",
    "BPI: Thank you for your payment of PHP 1,250.00 to MERALCO.",
    "Maya: You paid PHP 185.00 at Jollibee. Ref No. 88472910.",
] * 15  # Repeat slightly so local PH terms carry weight in TF-IDF
df_local_ham = pd.DataFrame({"text": ph_ham_samples, "label": 0})


# 4. Lightweight regex normalization (keep identical to iPhone JS later)
def clean_sms(text):
  text = str(text).lower()
  text = re.sub(r"http\S+|www\.\S+|bit\.ly/\S+", " urltoken ", text)
  text = re.sub(r"\b\d{4,}\b", " numtoken ", text)
  text = re.sub(r"\s+", " ", text).strip()
  return text


# Deduplicate external datasets FIRST
df_raw = pd.concat([df_ph_spam, df_uci_ham], ignore_index=True)
df_raw["clean_text"] = df_raw["text"].apply(clean_sms)
df_raw = df_raw[df_raw["clean_text"].str.len() > 2].drop_duplicates(
    subset=["clean_text"]
)

# Add local Philippine Ham examples AFTER deduplication so they carry weight
ph_ham_samples = [
    "Your GCash OTP is 482910. Never share this code with anyone.",
    (
        "You have received PHP 500.00 in your GCash account from JUAN DELA CRUZ."
        " Ref No. 918273645."
    ),
    (
        "You have sent PHP 250.00 to MARIA CLARA via GCash. Your new balance is"
        " PHP 1,420.00."
    ),
    "BDO Online: Your One-Time PIN is 918273. Valid for 5 minutes.",
    "BPI Mobile: Your OTP is 647382. Do not share this code with anyone.",
    "Shopee: Your parcel with tracking PH29384712 is out for delivery today.",
    (
        "Lazada: Rider will arrive soon. Please prepare exact amount PHP"
        " 349.00."
    ),
    "GrabFood: Your driver is on the way to your location.",
    (
        "Globe: Your Go+99 promo is now active. Enjoy 8GB all-access data for 7"
        " days."
    ),
    "Smart: You have used 80% of your GigaPower data allocation.",
    "Maya: You paid PHP 185.00 at Jollibee. Ref No. 88472910.",
    "UnionBank: Thank you for your payment of PHP 1,250.00 to MERALCO.",
] * 25

df_local_ham = pd.DataFrame({"text": ph_ham_samples, "label": 0})
df_local_ham["clean_text"] = df_local_ham["text"].apply(clean_sms)

df = pd.concat([df_raw, df_local_ham], ignore_index=True)
print(df["label"].value_counts().rename({0: "Ham (0)", 1: "Spam (1)"}))

Using Colab cache for faster access to the 'philippine-spam-sms-messages' dataset.
label
Ham (0)     4813
Spam (1)     930
Name: count, dtype: int64


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(class_weight="balanced", max_iter=500, random_state=42)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
print("=== Classification Report (Save for ACM Paper) ===")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

=== Classification Report (Save for ACM Paper) ===
              precision    recall  f1-score   support

         Ham       0.99      1.00      1.00       963
        Spam       0.99      0.96      0.98       186

    accuracy                           0.99      1149
   macro avg       0.99      0.98      0.99      1149
weighted avg       0.99      0.99      0.99      1149

Confusion Matrix:
 [[961   2]
 [  7 179]]


In [15]:
import json
import os
import numpy as np
import onnx
from onnx import TensorProto, helper
import onnxruntime as ort

num_features = X_train_tfidf.shape[1]

# 1. Export Logistic Regression weights using core ONNX ops (iOS Safari safe)
W = clf.coef_.T.astype(np.float32)
b = clf.intercept_.astype(np.float32)

X_in = helper.make_tensor_value_info(
    "float_input", TensorProto.FLOAT, [None, num_features]
)
Y_out = helper.make_tensor_value_info("prob_spam", TensorProto.FLOAT, [None, 1])

W_init = helper.make_tensor("W", TensorProto.FLOAT, W.shape, W.flatten().tolist())
b_init = helper.make_tensor("b", TensorProto.FLOAT, b.shape, b.flatten().tolist())

graph = helper.make_graph(
    [
        helper.make_node("MatMul", ["float_input", "W"], ["z"]),
        helper.make_node("Add", ["z", "b"], ["logits"]),
        helper.make_node("Sigmoid", ["logits"], ["prob_spam"]),
    ],
    "logreg_spam",
    [X_in],
    [Y_out],
    [W_init, b_init],
)

onnx_model = helper.make_model(
    graph,
    producer_name="maclrn",
    opset_imports=[helper.make_opsetid("", 13)],
)
onnx_model.ir_version = 8

with open("spam_model.onnx", "wb") as f:
  f.write(onnx_model.SerializeToString())

# 2. Export TF-IDF vocabulary & IDF weights to JSON
vocab_data = {
    "vocab": {k: int(v) for k, v in vectorizer.vocabulary_.items()},
    "idf": [float(x) for x in vectorizer.idf_],
    "num_features": int(num_features),
}
with open("vocab.json", "w") as f:
  json.dump(vocab_data, f)

# 3. Verify ONNX output
sess = ort.InferenceSession("spam_model.onnx")
sample_vec = X_test_tfidf[:2].toarray().astype(np.float32)
onnx_probs = sess.run(None, {"float_input": sample_vec})[0]
print("Verified! Sample Spam probabilities:", np.round(onnx_probs.flatten(), 4))

Verified! Sample Spam probabilities: [0.1462 0.8884]


In [16]:
from supabase import create_client

SUPABASE_URL = "https://rtidtpwhjfsmpkmcrnsr.supabase.co"
SUPABASE_SERVICE_KEY = "sb_secret_NLA4hQyYnzx6l22PDMpRWA_9oAMyAwC"

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

for filename, content_type in [
    ("spam_model.onnx", "application/octet-stream"),
    ("vocab.json", "application/json"),
]:
  with open(filename, "rb") as f:
    supabase.storage.from_("models").upload(
        path=filename,
        file=f,
        file_options={"content-type": content_type, "upsert": "true"},
    )

print("Uploaded spam_model.onnx and vocab.json to Supabase!")

Uploaded spam_model.onnx and vocab.json to Supabase!


In [19]:
import numpy as np
from onnx import TensorProto, helper
from scipy.sparse import csr_matrix, vstack

response = (
    supabase.table("flagged_spam").select("*").eq("processed", False).execute()
)
rows = response.data

if not rows:
  print("No new flagged spam vectors to process.")
else:
  new_vectors, row_ids = [], []
  for r in rows:
    row_ids.append(r["id"])
    sparse_dict = r["sparse_tfidf"]
    if not sparse_dict:
      continue
    cols = [int(k) for k in sparse_dict.keys()]
    vals = [float(v) for v in sparse_dict.values()]
    vec = csr_matrix((vals, ([0] * len(cols), cols)), shape=(1, num_features))
    new_vectors.extend([vec] * 25)

  X_updated = vstack([X_train_tfidf, vstack(new_vectors)])
  y_updated = np.concatenate(
      [y_train.to_numpy(), np.ones(len(new_vectors), dtype=int)]
  )

  clf.fit(X_updated, y_updated)

  # Export updated weights in iOS Safari-safe ONNX format
  W = clf.coef_.T.astype(np.float32)
  b = clf.intercept_.astype(np.float32)
  X_in = helper.make_tensor_value_info(
      "float_input", TensorProto.FLOAT, [None, num_features]
  )
  Y_out = helper.make_tensor_value_info("prob_spam", TensorProto.FLOAT, [None, 1])
  W_init = helper.make_tensor(
      "W", TensorProto.FLOAT, W.shape, W.flatten().tolist()
  )
  b_init = helper.make_tensor(
      "b", TensorProto.FLOAT, b.shape, b.flatten().tolist()
  )
  graph = helper.make_graph(
      [
          helper.make_node("MatMul", ["float_input", "W"], ["z"]),
          helper.make_node("Add", ["z", "b"], ["logits"]),
          helper.make_node("Sigmoid", ["logits"], ["prob_spam"]),
      ],
      "logreg_spam",
      [X_in],
      [Y_out],
      [W_init, b_init],
  )
  onnx_model = helper.make_model(
      graph,
      producer_name="maclrn",
      opset_imports=[helper.make_opsetid("", 13)],
  )
  onnx_model.ir_version = 8

  with open("spam_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

  with open("spam_model.onnx", "rb") as f:
    supabase.storage.from_("models").upload(
        path="spam_model.onnx",
        file=f,
        file_options={
            "content-type": "application/octet-stream",
            "upsert": "true",
        },
    )

  supabase.table("flagged_spam").update({"processed": True}).in_(
      "id", row_ids
  ).execute()
  print(
      f"Retrained with {len(rows)} flagged vector(s) and published updated"
      " spam_model.onnx!"
  )

Retrained with 1 flagged vector(s) and published updated spam_model.onnx!
